# 怎样训练一个评论情感分类器？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{
  "tfidf_accuracy": 0.75,
  "embedding_accuracy": 0.625,
  "test_samples": 8,
  "vocabulary_size": 77,
  "training_loss": 0.00035102423862554133,
  "elapsed_seconds": 4.146
}


## 1. 补全：未知词比例
计算测试语料中未知词占全部词的比例，检查分母非零，打印有哪些未知词。

In [2]:
tokens = [w for t in test_texts for w in tokenize(t)]
unknown = [w for w in tokens if w not in vocab]
ratio = len(unknown) / len(tokens)
assert 0 <= ratio <= 1
print(ratio, sorted(set(unknown)))

0.10256410256410256 ['a', 'all', 'at', 'but']


## 2. 修改：补齐不应改变结果
同一句话编码到长度 10 和 20，检查掩码平均产生的分数一致。

In [3]:
with torch.no_grad():
    a = network(encode(['good food'], 10))
    b = network(encode(['good food'], 20))
assert torch.allclose(a, b, atol=1e-06)
print(a)

tensor([[-6.2046,  6.4205]])


## 3. 迁移：另建中文词级基线
用分好词的中文服务评价训练一个新分类器，不混用英文词表；核对正负对照。

In [4]:
cn = ['服务 很 好', '态度 友好 推荐', '服务 很 差', '态度 粗鲁 失望']
cv = TfidfVectorizer(tokenizer=str.split, token_pattern=None)
cl = LogisticRegression(C=10).fit(cv.fit_transform(cn), [1, 1, 0, 0])
cp = cl.predict(cv.transform(['友好 推荐', '粗鲁 失望']))
assert cp.tolist() == [1, 0]
print(cp)

[1 0]
